# 146 — Experimentos, semillas y trazabilidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** A: media = (0.790+0.798+0.794)/3 = **0.794**, desviación muestral =
0.004. B: media = (0.803+0.775+0.792)/3 = **0.790**, desviación ≈ 0.014. A tiene media
levemente mayor y es **mucho más estable**; la diferencia de medias (0.004) está dentro
del ruido de B (±0.014), así que la evidencia a favor de A es débil pero su menor
varianza es un criterio legítimo de desempate. Con solo 3 semillas, lo honesto es: «A
preferible por estabilidad; la diferencia de medias no es concluyente».

**Ejercicio 2.** `run_b` no es relanzable: le falta la **semilla** y su dataset es una
referencia viva («tabla_clientes») sin snapshot ni hash — dos entradas sin identidad.
Nótese que es justamente el run con mejor métrica (0.84): los runs irreproducibles con
buena métrica son los más peligrosos, porque invitan a promover algo que no se puede
reconstruir.

**Ejercicio 3.** Con la misma semilla el resultado es idéntico (repetibilidad); con
`seed=147` cambian los valores muestreados pero no el contrato (`kind`, estructura de
`evidence`). Ilustra que la semilla controla el azar propio del programa, y que comparar
corridas exige fijarla o promediar sobre varias.

**Ejercicio 4.** Aristas: `datos_crudos → ds-2025-12`, `ds-2025-12 → features_v3`,
`features_v3 → run_0042`, `commit a1b2c3 → run_0042`, `seed 42 → run_0042`,
`run_0042 → churn-v7`. Si cambia `features_v3` quedan obsoletos sus descendientes:
`run_0042` y `churn-v7` (análisis de impacto = alcance hacia adelante); `ds-2025-12` y
`datos_crudos` no se ven afectados.


In [ ]:
result = run_lab("observability", seed=146)
assert result["kind"] == "observability"
assert result["evidence"]
show(result)


In [ ]:
import statistics

A = [0.790, 0.798, 0.794]
B = [0.803, 0.775, 0.792]
for name, xs in [("A", A), ("B", B)]:
    print(f"{name}: media={statistics.mean(xs):.4f}  desv={statistics.stdev(xs):.4f}")

# Ejercicio 3: repetibilidad con la misma semilla
r1 = run_lab("observability", seed=146)
r2 = run_lab("observability", seed=146)
r3 = run_lab("observability", seed=147)
print("misma semilla, mismo resultado:", r1 == r2)
print("semilla distinta, mismo contrato:", r1["kind"] == r3["kind"])


## Reflexión

1. Fijaste `seed=146` y dos corridas en máquinas distintas dieron métricas diferentes: enumera al menos tres causas posibles y el control que neutraliza cada una.
2. ¿Por qué reportar el mejor run entre 10 semillas es metodológicamente equivalente a evaluar en el conjunto de entrenamiento? ¿Qué estadístico reportarías en su lugar?
3. Si el hash del snapshot de datos no se hubiera registrado en el run, ¿qué preguntas de linaje (procedencia e impacto) se vuelven incontestables?
